In [35]:
suppressMessages(library(gscreend))
suppressMessages(library(ggplot2))
suppressMessages(library(SummarizedExperiment))

In [41]:
raw_counts <- read.csv("/scratch/peidli/SYMPHONY/pooled_screen_1/pooled_screen_1_counts.csv")
counts_matrix <- raw_counts[, 16:ncol(raw_counts)]  # select only the counts, only last Pool! TODO: change to all pools averaged!

In [42]:
head(raw_counts)

,unique_guide_name,Transcript_ID,ensembl_gene_id,Gene,transcript_biotype,GuideName,GuideSeq,MatchPos,Guide_Score,Region,⋯,sym.03.02.TP2,sym.03.02.TP3,sym.03.02.TP4,sym.03.02.TP5,sym.03.02.TP6,sym.03.03.TP1,sym.03.03.TP2,sym.03.03.TP4,sym.03.03.TP5,sym.03.03.TP6
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,AATBC_ENST00000400385_1005,ENST00000400385,ENSG00000215458,AATBC,lncRNA,crRNA0983:983-1005,ACAGAACCCTAGGAAGCAACCTC,1005,0.7896259,Gene_Body,⋯,1235,1494,1747,1655,1494,1545,1298,1842,1277,1391
2,AATBC_ENST00000400385_1562,ENST00000400385,ENSG00000215458,AATBC,lncRNA,crRNA1504:1540-1562,CCAGGCCACTTGCTCCACCCTCT,1562,0.7478140,Gene_Body,⋯,1885,2072,2450,2161,2318,2164,1775,2276,1863,2168
3,AATBC_ENST00000400385_1938,ENST00000400385,ENSG00000215458,AATBC,lncRNA,crRNA1862:1916-1938,TGACCTCTCACCCCGCAGCCTCT,1938,0.7476817,Gene_Body,⋯,1199,1299,1680,1402,1473,1463,1166,1650,1139,1391
4,AATBC_ENST00000400385_2713,ENST00000400385,ENSG00000215458,AATBC,lncRNA,crRNA2601:2691-2713,CTGCCTTATAAAGATAGACCGGT,2713,0.8705397,Gene_Body,⋯,799,1175,1268,1071,1026,1157,892,1263,835,1073
5,AATBC_ENST00000400385_28,ENST00000400385,ENSG00000215458,AATBC,lncRNA,crRNA0006:6-28,CCGGTCTCTGCAGTCACCCTCCT,28,0.7625627,Gene_Body,⋯,1518,1898,2201,1797,1975,2039,1678,2248,1576,1842
6,AATBC_ENST00000400385_3041,ENST00000400385,ENSG00000215458,AATBC,lncRNA,crRNA2929:3019-3041,ATGCTCAAATATGACCGTGACCT,3041,0.8810543,Gene_Body,⋯,1317,1520,1730,1563,1695,1584,1317,1791,1320,1462


In [43]:
names <- colnames(counts_matrix)
names

[1] "Sym.pool.1.3"  "sym.01.01.TP1" "sym.01.01.TP2" "sym.01.01.TP3"
 [5] "sym.01.01.TP4" "sym.01.01.TP5" "sym.01.01.TP6" "sym.01.02.TP1"
 [9] "sym.01.02.TP2" "sym.01.02.TP3" "sym.01.02.TP4" "sym.01.02.TP5"
[13] "sym.01.02.TP6" "sym.01.03.TP1" "sym.01.03.TP2" "sym.01.03.TP3"
[17] "sym.01.03.TP4" "sym.01.03.TP5" "sym.01.03.TP6" "sym.02.01.TP1"
[21] "sym.02.01.TP2" "sym.02.01.TP3" "sym.02.01.TP4" "sym.02.01.TP5"
[25] "sym.02.01.TP6" "sym.02.02.TP1" "sym.02.02.TP2" "sym.02.02.TP3"
[29] "sym.02.02.TP4" "sym.02.02.TP5" "sym.02.02.TP6" "sym.02.03.TP1"
[33] "sym.02.03.TP2" "sym.02.03.TP3" "sym.02.03.TP4" "sym.02.03.TP5"
[37] "sym.02.03.TP6" "sym.03.01.TP1" "sym.03.01.TP2" "sym.03.01.TP3"
[41] "sym.03.01.TP4" "sym.03.01.TP5" "sym.03.01.TP6" "sym.03.02.TP1"
[45] "sym.03.02.TP2" "sym.03.02.TP3" "sym.03.02.TP4" "sym.03.02.TP5"
[49] "sym.03.02.TP6" "sym.03.03.TP1" "sym.03.03.TP2" "sym.03.03.TP4"
[53] "sym.03.03.TP5" "sym.03.03.TP6"

In [72]:
# split names by "."
x <- strsplit(names, "\\.")
cell_lines <- sapply(x, function(x) x[2])
# third element of each entry is the replicate
replicates <- sapply(x, function(x) x[3])
# last element of each entry is the time point; remove the "P" character and keep the rest
time_points <- sapply(x, function(x) gsub("P", "", x[4]))
# time_points[1:3] = "TP0"
time_points[2:length(time_points)] = 'T1'
time_points[1] = "T0"

In [73]:
# guide annotation
rowData <- data.frame(
    sgRNA_id = raw_counts$unique_guide_name,
    gene = raw_counts$Gene,
    Transcript_ID = raw_counts$Transcript_ID
)

# sample annotation
names <- colnames(counts_matrix)
colData <- data.frame(
    samplename = names,
    # timepoint naming convention: 
    # T0 -> reference, 
    # T1 -> selected
    timepoint = time_points,
    cell_line = cell_lines,
    replicate = replicates
)

se <- SummarizedExperiment(assays=list(counts=counts_matrix),
                        rowData=rowData, colData=colData)

In [74]:
# look at coldata
colData(se)

DataFrame with 54 rows and 4 columns
                 samplename   timepoint   cell_line   replicate
                <character> <character> <character> <character>
Sym.pool.1.3   Sym.pool.1.3          T0        pool           1
sym.01.01.TP1 sym.01.01.TP1          T1          01          01
sym.01.01.TP2 sym.01.01.TP2          T1          01          01
sym.01.01.TP3 sym.01.01.TP3          T1          01          01
sym.01.01.TP4 sym.01.01.TP4          T1          01          01
...                     ...         ...         ...         ...
sym.03.03.TP1 sym.03.03.TP1          T1          03          03
sym.03.03.TP2 sym.03.03.TP2          T1          03          03
sym.03.03.TP4 sym.03.03.TP4          T1          03          03
sym.03.03.TP5 sym.03.03.TP5          T1          03          03
sym.03.03.TP6 sym.03.03.TP6          T1          03          03

In [75]:
# but I have set one sample to T0
pse <- createPoolScreenExp(se)

Creating PoolScreenExp object from a SummarizedExperiment object.

References and samples are named correctly.

Data concerning sgRNA and genes is provided.



In [76]:
pse <- RunGscreend(pse)

ERROR: Error in `assays<-`(`*tmp*`, value = `*vtmp*`): please use 'assay(x, withDimnames=FALSE)) <- value' or 'assays(x,
  withDimnames=FALSE)) <- value' when the rownames or colnames of the
  supplied assay(s) are not identical to those of the receiving
  SummarizedExperiment object 'x'
